# Hate Speech Classification Assignment (RNN)


## Load Data


In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('imbalanced_data.csv')
data.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [3]:
df = data.drop(columns=['id'])
df.head()

,label,tweet
0,0,@user when a father is dysfunctional and is s...
1,0,@user @user thanks for #lyft credit i can't us...
2,0,bihday your majesty
3,0,#model i love u take with u all the time in ...
4,0,factsguide: society now #motivation


## Text Cleaning


In [4]:
df['tweet'] = df['tweet'].str.lower()
df['tweet'].head()

0     @user when a father is dysfunctional and is s...
1    @user @user thanks for #lyft credit i can't us...
2                                  bihday your majesty
3    #model   i love u take with u all the time in ...
4               factsguide: society now    #motivation
Name: tweet, dtype: object

In [5]:
df['tweet'] = df['tweet'].str.replace(r'(@\S+)|(https?://\S+)|([^a-zA-Z\s])', ' ', regex=True)
df['tweet'].head()

0       when a father is dysfunctional and is so se...
1        thanks for  lyft credit i can t use cause ...
2                                  bihday your majesty
3     model   i love u take with u all the time in ...
4               factsguide  society now     motivation
Name: tweet, dtype: object

In [6]:
df['tweet'] = df['tweet'].str.strip()
df['tweet'].head()

0    when a father is dysfunctional and is so selfi...
1    thanks for  lyft credit i can t use cause they...
2                                  bihday your majesty
3      model   i love u take with u all the time in ur
4               factsguide  society now     motivation
Name: tweet, dtype: object

## Tokenization & Vocabulary


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
vocab_size = 5000
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token = "<OOV"
)

In [10]:
tokenizer.fit_on_texts(df['tweet'])

## Text to Sequences


In [12]:
sequences = tokenizer.texts_to_sequences(df['tweet'])

In [26]:
lengths = [len(seq) for seq in sequences]
import numpy as np
print(np.percentile(lengths, [50, 75, 90, 95, 99]))

[12. 16. 20. 22. 26.]


In [27]:
padded = pad_sequences(sequences,maxlen=22,padding='post',truncating='post')

In [28]:
word_index = tokenizer.word_index

print("Word Index:", word_index)
print("Sequences:", sequences)
print("Padded:", padded)

Word Index: {'<OOV': 1, 'the': 2, 'to': 3, 'i': 4, 'a': 5, 'you': 6, 'and': 7, 'in': 8, 'for': 9, 'of': 10, 'is': 11, 'my': 12, 'it': 13, 's': 14, 'love': 15, 'this': 16, 'on': 17, 'with': 18, 'be': 19, 't': 20, 'day': 21, 'that': 22, 'all': 23, 'so': 24, 'are': 25, 'me': 26, 'amp': 27, 'happy': 28, 'your': 29, 'at': 30, 'have': 31, 'we': 32, 'am': 33, 'can': 34, 'just': 35, 'will': 36, 'when': 37, 'not': 38, 'do': 39, 'u': 40, 'like': 41, 'life': 42, 'what': 43, 'time': 44, 'm': 45, 'but': 46, 'today': 47, 'from': 48, 'up': 49, 'now': 50, 'new': 51, 'thankful': 52, 'out': 53, 'as': 54, 'positive': 55, 'get': 56, 'was': 57, 'people': 58, 'bihday': 59, 'good': 60, 'about': 61, 'how': 62, 'our': 63, 'by': 64, 'no': 65, 'they': 66, 'one': 67, 'see': 68, 'smile': 69, 'more': 70, 'if': 71, 'don': 72, 'go': 73, 'who': 74, 'father': 75, 'want': 76, 'he': 77, 'take': 78, 'work': 79, 'fun': 80, 'healthy': 81, 'weekend': 82, 're': 83, 'summer': 84, 'an': 85, 'has': 86, 'there': 87, 'or': 88, 'ma

In [29]:
X = padded
y = df['label'].values

## Train-Test Split

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [32]:
X_train

array([[ 159,   17, 1501, ...,   31,  154,    0],
       [  92,   86,  451, ...,    0,    0,    0],
       [ 858,   49,   27, ..., 3175, 2098,    0],
       ...,
       [   4, 3589,  186, ...,    0,    0,    0],
       [4914,    1, 1071, ...,    0,    0,    0],
       [  12, 1758,   14, ...,    0,    0,    0]],
      shape=(25569, 22), dtype=int32)

## Model Building


In [33]:
from tensorflow import keras

In [37]:
model_rnn = keras.models.Sequential()
model_rnn.add(keras.layers.Embedding(input_dim=10000,output_dim=128,input_length=40))
model_rnn.add(keras.layers.SimpleRNN(64,return_sequences=False))
model_rnn.add(keras.layers.Dense(128,activation='relu'))
model_rnn.add(keras.layers.Dropout(0.4))
model_rnn.add(keras.layers.Dense(1,activation='sigmoid'))

model_rnn.build(input_shape=(None,40))
model_rnn.summary()

c:\Users\admin\miniconda3\envs\base_ml\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,300,801 (4.96 MB)

 Trainable params: 1,300,801 (4.96 MB)

 Non-trainable params: 0 (0.00 B)

## Training

In [38]:
model_rnn.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy']
                  )

In [39]:
history = model_rnn.fit(X_train,
                        y_train,
                        epochs=10,
                        batch_size=32
                        )

Epoch 1/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9295 - loss: 0.2434
Epoch 2/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9380 - loss: 0.2057
Epoch 3/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9597 - loss: 0.1219
Epoch 4/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9732 - loss: 0.0806
Epoch 5/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9833 - loss: 0.0539
Epoch 6/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9880 - loss: 0.0398
Epoch 7/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9898 - loss: 0.0340
Epoch 8/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9892 - loss: 0.0322
Epoch 9/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9926 - loss: 0.0242
Epoch 10/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9944 - loss: 0.0182


## Evaluation


In [40]:
pred = model_rnn.predict(X_test)

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [44]:
pred_labels = (pred > 0.5).astype(int)
pred_labels.flatten()

array([0, 0, 0, ..., 0, 0, 0], shape=(6393,))

In [45]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, pred_labels))
print(classification_report(y_test, pred_labels))

Accuracy: 0.9479117785077429
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      5942
           1       0.66      0.54      0.60       451

    accuracy                           0.95      6393
   macro avg       0.81      0.76      0.78      6393
weighted avg       0.94      0.95      0.95      6393

